# Knowledge Distillation — ResNet-50 to a Compact CNN

A small CNN is trained twice on the same images, with the same architecture, the same
optimiser and the same schedule. The only difference is what it is asked to match:

* the **hard one-hot label**, or
* the **soft class distribution** a fine-tuned ResNet-50 produces.

On CIFAR-10, with a teacher 6.8 points clear of the student, that change is worth
**+0.04 points**. Which is to say: nothing. This notebook is the honest account of why,
and of the two experiments that established it was not an accident.

| Model | Parameters | Test accuracy |
|---|---:|---:|
| ResNet-50 teacher | 23,528,522 | 90.03% |
| Student CNN — baseline | 288,746 | 83.26% |
| Student CNN — distilled | 288,746 | 83.30% |

**How to read this notebook.** The training itself lives in `distill.py` and takes about
two hours on a CPU. This notebook walks through the method, shows the exact code that
produced the numbers, and redraws the figures from the `results.json` that run wrote. It
does not retrain anything — so every number below is the measured one, not a fresh
approximation of it.

## 1. The objective

Hinton, Vinyals & Dean (2015) train the student against the teacher's softened output
rather than the label:

$$\mathcal{L} = \alpha \cdot T^2 \cdot \mathrm{KL}\!\left(\sigma(z_s/T) \,\|\, \sigma(z_t/T)\right) + (1-\alpha)\cdot \mathrm{CE}(z_s, y)$$

Two things in that line are easy to skip past and both matter.

**Why soft targets carry more than labels.** A one-hot label says an image is a cat. The
teacher says it is 0.82 cat, 0.11 dog, 0.005 airplane. The relative weight on the *wrong*
classes is a statement about which classes resemble each other, and the student picks that
structure up from every image — not only from the ones it gets wrong.

**Why the $T^2$.** Dividing the logits by $T$ shrinks the gradient of the soft term by
roughly $1/T^2$. Without the correction, raising the temperature would quietly also lower
the learning rate on that term, and $\alpha$ would stop meaning the same thing from one
temperature to the next. The $T^2$ restores the scale, so $\alpha$ is the only knob that
controls the balance between the two terms.

In [ ]:
import torch
import torch.nn.functional as F


def distillation_loss(student_logits, teacher_logits, targets, T=4.0, alpha=0.7):
    """The whole method, in six lines."""
    soft = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),   # student, as log-probabilities
        F.softmax(teacher_logits / T, dim=1),       # teacher, as probabilities
        reduction="batchmean",
    ) * (T ** 2)                                    # undo the 1/T^2 gradient shrink
    hard = F.cross_entropy(student_logits, targets)
    return alpha * soft + (1 - alpha) * hard

### What the temperature actually does

Nothing here is trained — this is just the softmax at three temperatures, on one plausible
set of teacher logits, to make concrete what "softer" means.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

classes = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
logits = torch.tensor([[1.1, 0.4, 2.2, 6.8, 1.6, 5.1, 0.9, 1.4, 0.7, 0.3]])  # a "cat" image

fig, ax = plt.subplots(figsize=(8, 3.2))
x = np.arange(len(classes))
for T, colour in [(1.0, "#94a3b8"), (4.0, "#f59e0b"), (10.0, "#e11d48")]:
    p = F.softmax(logits / T, dim=1).numpy().ravel()
    ax.plot(x, p, "o-", ms=4, color=colour, label=f"T = {T:g}")
ax.set_xticks(x); ax.set_xticklabels(classes, rotation=30)
ax.set_ylabel("probability"); ax.legend(frameon=False)
ax.set_title("The same teacher logits at three temperatures")
ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

At $T=1$ the teacher is almost one-hot and says little the label did not. At $T=4$ the
ranking of the wrong classes — dog above deer above bird above plane — becomes visible
signal. At $T=10$ the distribution flattens towards uniform and that ranking starts to
wash out again. $T=4$ is the value used throughout.

## 2. The two models

**Teacher — ResNet-50 with its stem left alone.** The standard CIFAR adaptation replaces the
7×7 stride-2 stem and drops the max-pool, so a 32×32 image is not immediately reduced to
8×8. Tried first, that was a mistake: the replacement stem is randomly initialised, so every
pretrained block downstream receives features it has never seen. That teacher reached
82.1% — *below* the 83.9% of the student it was meant to teach — and distilling from it
**cost 3.0 points**.

Keeping the stem and upsampling CIFAR to 64×64 instead lets every pretrained weight do
the job it was trained for, and is ~2.8× cheaper per step. Teacher: 82.1% → **90.0%**.
`layer1`/`layer2` stay frozen.

**Student — three conv blocks.** 288,746 parameters, about
1.2% of the teacher's 23,528,522.

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50


def build_teacher():
    model = resnet50(weights="IMAGENET1K_V1")   # stem untouched; inputs upsampled to 64x64
    model.fc = nn.Linear(2048, 10)
    for name, param in model.named_parameters():
        if name.startswith(("layer1", "layer2")):
            param.requires_grad = False
    return model


class StudentCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.1), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


student = StudentCNN()
print(f"student: {sum(p.numel() for p in student.parameters()):,} parameters")

## 3. What makes the comparison a comparison

Everything about the two student runs is held fixed except the loss:

* same class, same seed, same initialisation
* same augmentation (random crop with padding 4, horizontal flip)
* same optimiser (AdamW, lr 2e-3, weight decay 5e-4) and the same 20-epoch cosine schedule
* same images in the same order

**Teacher logits are cached.** One forward pass over the training images before the student
runs, stored and indexed alongside the dataset. Distillation then costs no more per epoch
than baseline training does. The teacher scored the *un-augmented* image, which is the
standard choice: the target stays stable across epochs instead of jittering with the crop.

**The training set is a fixed 20,000-image subset** of CIFAR-10's 50,000, drawn
once with a seeded generator and reused by every run. That is a CPU budget, not a design
decision, and it is worth stating plainly: distillation helps *more* when data is scarce,
so the gain here is probably an upper estimate of what the full training set would give.

## 4. Results

Loaded from the `results.json` that `distill.py` wrote.

In [ ]:
import json
from pathlib import Path

R = json.load(open("results.json"))
t = R["teacher_acc"] * 100
b = R["baseline_acc"] * 100
d = R["distilled_acc"] * 100

print(f"teacher   ResNet-50    {t:6.2f}%   {R['teacher_params']:>10,} params")
print(f"student   baseline     {b:6.2f}%   {R['student_params']:>10,} params")
print(f"student   distilled    {d:6.2f}%   {R['student_params']:>10,} params")
print(f"\ndistillation gain     {d - b:+6.2f} points")
print(f"compression           {R['compression']:.0f}x fewer parameters")
print(f"teacher gap closed    {(d - b) / (t - b) * 100:.0f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
bh, dh = R["baseline_history"], R["distilled_history"]
ep = [h["epoch"] for h in bh]

ax[0].plot(ep, [h["test_acc"] * 100 for h in bh], "o-", ms=3, color="#64748b", label="baseline (hard labels)")
ax[0].plot(ep, [h["test_acc"] * 100 for h in dh], "o-", ms=3, color="#e11d48", label="distilled")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("test accuracy (%)")
ax[0].set_title("Student accuracy during training"); ax[0].legend(frameon=False)

ax[1].plot(ep, [h["loss"] for h in bh], "o-", ms=3, color="#64748b", label="cross-entropy")
ax[1].plot(ep, [h["loss"] for h in dh], "o-", ms=3, color="#e11d48", label="KD objective")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("training loss")
ax[1].set_title("Training loss — different objectives,\nnot comparable to each other")
ax[1].legend(frameon=False)

for a in ax:
    a.grid(alpha=0.3); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

The distilled student is ahead for essentially the whole run, not only at the end — this is
not a late-schedule artifact.

The loss panel is there to be read carefully: the two curves are **different objectives**
and their absolute values say nothing about which model is better. Only the accuracy panel
answers that.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
names = ["ResNet-50\nteacher", "student\nbaseline", "student\ndistilled"]
accs = [t, b, d]
cols = ["#0ea5e9", "#64748b", "#e11d48"]

bars = ax[0].bar(names, accs, color=cols, width=0.55)
for bar, a in zip(bars, accs):
    ax[0].text(bar.get_x() + bar.get_width() / 2, a + 0.6, f"{a:.1f}%", ha="center", fontsize=9)
ax[0].set_ylabel("test accuracy (%)"); ax[0].set_ylim(0, max(accs) + 8)
ax[0].set_title(f"Accuracy on the full {R['n_test']:,}-image test set")

params = [R["teacher_params"] / 1e6, R["student_params"] / 1e6, R["student_params"] / 1e6]
ax[1].scatter(params, accs, s=110, c=cols, zorder=3)
for x_, y_, n in zip(params, accs, ["teacher", "baseline", "distilled"]):
    ax[1].annotate(n, (x_, y_), textcoords="offset points", xytext=(8, -3), fontsize=9)
ax[1].annotate("", xy=(params[2], accs[2]), xytext=(params[1], accs[1]),
               arrowprops=dict(arrowstyle="->", color="#e11d48", lw=1.6))
ax[1].set_xscale("log"); ax[1].set_xlim(0.1, 200)
ax[1].set_xlabel("parameters (millions, log scale)"); ax[1].set_ylabel("test accuracy (%)")
ax[1].set_title("Distillation moves the student up\nat an unchanged parameter count")

for a in ax:
    a.grid(alpha=0.3); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 5. The alpha sweep

alpha = 0.7 puts most of the weight on matching a distribution a 288,746-parameter
network can only partly represent. Worth testing rather than assuming — so sweep it, with a
protocol that makes the answer mean something:

* **2,000 of the 20,000 training images held out as validation.** Nothing trains on them.
* every run trains on the same 18,000 images, same seed, same schedule, same augmentation.
* **alpha is chosen on validation accuracy**, and only the chosen model is then scored on
  test. Choosing by test accuracy and reporting that number is how a sweep becomes a lie.

| Run | Validation | Test |
|---|---:|---:|
| baseline (no KD) | 82.45% | 83.06% |
| distilled, alpha = 0.3 | 83.00% | 83.14% |
| distilled, alpha = 0.5 | 83.05% | 83.21% |
| distilled, alpha = 0.7 | 83.05% | 83.19% |
| distilled, alpha = 0.9 | 83.05% | 82.70% |

Validation picked **alpha = 0.5**, scoring **83.21%** on test
against the baseline's 83.06% — **+0.15 points**. Four values of alpha,
all within half a point of the baseline. alpha is not the problem.

## 6. So what is the problem

Two things in this setup are known to break distillation, and both are true here.

**The student cannot represent the target.** The teacher's distribution over 10 classes
encodes structure that a three-block CNN with a 128-wide penultimate layer has no capacity
to reproduce. Past some point the KL term is asking for something unreachable — and
weighting it harder (alpha = 0.9, the one run that came out *worse*) just spends more of a
fixed budget on it.

**The teacher never saw what the student sees.** The cached logits were computed on the
clean image; the student trains on a random crop and a 50% horizontal flip. Roughly half
the time the student is told to match the teacher's answer for a picture it is not looking
at. Beyer et al. (2022) make exactly this the central point: distillation works when
teacher and student see *identical* views, and it needs long schedules to pay off. This
implementation satisfies neither — the most likely reason the gain is flat.

The conclusion is not "distillation does not work". It is that **a teacher with a real
margin is necessary but not sufficient**: experiment 1 shows what happens without the
margin (-3.0 points), and experiments 2 and 3 show the margin alone buys nothing.

### What would be tried next

1. **Consistent teaching** — run the teacher live on the same augmented view instead of
   caching clean logits. Costs a teacher forward pass per batch; the most likely fix.
2. **Longer schedules** — the distilled runs were still improving at epoch 20 while the
   baseline had plateaued.
3. **More student capacity** — a wider penultimate layer, so there is something to
   distil into.

## References

* Hinton, G., Vinyals, O., Dean, J. (2015). *Distilling the Knowledge in a Neural Network.* [arXiv:1503.02531](https://arxiv.org/abs/1503.02531)
* Beyer, L. et al. (2022). *Knowledge Distillation: A Good Teacher is Patient and Consistent.* [arXiv:2106.05237](https://arxiv.org/abs/2106.05237)
* He, K., Zhang, X., Ren, S., Sun, J. (2016). *Deep Residual Learning for Image Recognition.* [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)